In [1]:
#设置并加载秘钥
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 设置langsmith
import os


In [3]:
#处理文档
import bs4
import requests
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_web_page(url: str, bs_kwargs: dict | None = None) -> list[Document]:
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    soup = bs4.BeautifulSoup(response.text, "html.parser", **(bs_kwargs or {}))
    return [Document(page_content=soup.get_text(), metadata={"source": url})]


urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

docs = [load_web_page(url) for url in urls]

In [4]:
# 分割文档
from langchain_text_splitters import RecursiveCharacterTextSplitter
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 100,
    chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)

In [5]:
# 文本向量化
MILVUS_URI = "http://localhost:19530"
DB_NAME = "agentic_arg"
COLLECTION_NAME="url"

EMBED_MODEL_NAME = "Pro/BAAI/bge-m3"
EMBED_MODEL_DIM = 1024
# 构建数据库
## 初始化客户端
from http import client
from typing import Collection
from pymilvus import MilvusClient
client = MilvusClient(MILVUS_URI)
existed_databases = client.list_databases()
if DB_NAME not in existed_databases:
    client.create_database(db_name=DB_NAME)

client.use_database(db_name=DB_NAME)

## 创建collection
if client.has_collection(collection_name=COLLECTION_NAME):
    client.drop_collection(collection_name=COLLECTION_NAME)
# 创建指定的collection
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBED_MODEL_DIM,
    metric_type='COSINE'
)
# 初始化编码模型
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv(override=True)
# 初始化大模型
embed_model = OpenAIEmbeddings(
    model=EMBED_MODEL_NAME,
    api_key = os.getenv("GUIJI_API_KEY"),
    base_url = os.getenv("GUIJI_BASE_URL")
)

In [6]:
# 向量化：
texts = [chunk.page_content for chunk in doc_splits]
vectors = embed_model.embed_documents(texts)
# 构建数据
data = [
    {
        "id":i,
        "vector":vectors[i],
        "text":doc_splits[i].page_content,
        "source":doc_splits[i].metadata.get("source","unknown"),
        "chunk_id":i
    }
    for i in range(len(doc_splits))
]
# 写入milvus
insert_res = client.upsert(
    collection_name=COLLECTION_NAME,
    data=data
)
print(f"成功插入{insert_res['upsert_count']}条记录")
# 持久化
client.flush(collection_name=COLLECTION_NAME)

成功插入538条记录


In [8]:
# 封装检索器函数，
from functools import lru_cache
@lru_cache(maxsize=1)
def _get_retriever():
    """返回一个检索函数，用于从Milvus检索相关文档"""
    def retrieve(query:str,k:int=3):
        # 向量化查询
        query_vector = embed_model.embed_query(query)
        # 从milvus检索
        results = client.search(
            collection_name = COLLECTION_NAME,
            data=[query_vector],
            limit=k,
            output_fields = ["text","source","chunk_id"]
        )
        # 转化为langchain Document格式
        from langchain_core.documents import Document
        docs = [
            Document(
                page_content=hit["entity"]["text"],
                metadata={
                    "source":hit["entity"]["source"],
                    "chunk_id":hit["entity"]["chunk_id"],
                    "score":hit["distance"]
                }
            )
            for hit in results[0]
        ]
        return docs
    return retrieve


In [9]:
# 测试检索
retriever = _get_retriever()
test_query = "What is reward hacking?"
results = retriever(test_query)

print(f"检索到 {len(results)} 条结果：\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] Score: {doc.metadata['score']:.4f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content[:200]}...\n")

检索到 3 条结果：

[1] Score: 0.5765
Source: https://lilianweng.github.io/posts/2024-11-28-reward-hacking/
Content: a big goal into small goals? Is the reward sparse or dense? How you measure the success? Various choices may lead to good or problematic learning dynamics, including unlearnable tasks or hackable rewa...

[2] Score: 0.5565
Source: https://lilianweng.github.io/posts/2024-11-28-reward-hacking/
Content: Most of the past work on this topic has been quite theoretical and focused on defining or demonstrating the existence of reward hacking. However, research into practical mitigations, especially in the...

[3] Score: 0.5549
Source: https://lilianweng.github.io/posts/2024-11-28-reward-hacking/
Content: Let’s Define Reward Hacking#
Reward shaping in RL is challenging. Reward hacking occurs when an RL agent exploits flaws or ambiguities in the reward function to obtain high rewards without genuinely l...

